In [1]:
# 식품 영양 정보를 가져와서 csv 파일로 저장
# 가져올 정보 : 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리

# 필요한 패키지
import requests
import pandas as pd


In [2]:
# 공공데이터 url = End Point + 상세기능 url
url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'

serviceKey = 'f9b40d8c2d951e1c473b6fa9523a0bb4db23f460964ca63df9e0585cf54473ac'
page = 1
params = {
  'serviceKey' : serviceKey, # 필수, 나머지는 선택
  'pageNo' : page,
  'numOfRow' : 10,
  'type' : 'json',
}

response = requests.get(url, params=params)

try:
  response.raise_for_status()


except Exception as e:
  print(f'예외 발생 : {e}')

In [3]:
items = response.json()['body']['items']
list = [] # 식품 정보를 관리할 리스트
for item in items:
  # 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리
  dic = {
    '식품명' : item['FOOD_NM_KR'],
    '탄수화물(g)' : item['AMT_NUM6'],
    '지방(g)' : item['AMT_NUM4'],
    '단백질(g)' : item['AMT_NUM3'],
    '나트륨(mg)' : item['AMT_NUM13'],
    '총칼로리(kal)' : item['AMT_NUM1'],
    '기준량' : item['SERVING_SIZE'],
  }
  list.append(dic)
 

  # 받아온 리스트를 데이터프레임으로 변환
  df = pd.DataFrame(list)
  

In [4]:
# 중복제거
# df = df.drop_duplicates() # 행 전체 값이 중복되면 제거
# df = df.drop_duplicates(subset=['식품명']), # 식품명 열이 중복된 경우만 제거
df = df.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거
# df = df.drop_duplicates(subset=['식품명'], keep='last') # 중복되면 마지막을 살리고 나머지 제거
# df = df.drop_duplicates(subset=['식품명'], keep=False) # 중복되면 모두 제거

# 결측치 값 변경
df['탄수화물(g)'] = df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경


# 결측치 제거
df = df.dropna() # 행에 결측치가 하나라도 있으면 삭제
df = df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제

In [5]:
serviceKey = 'f9b40d8c2d951e1c473b6fa9523a0bb4db23f460964ca63df9e0585cf54473ac'

def get_food_nutrient(serviceKey:str, page:int=1)->pd.DataFrame:
  """공공데이터에서 식품영양소 db 정보를 가져와서 데이터 프레임으로 반환하는 함수"""
  url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'
  params = {
        'serviceKey' : serviceKey,
        'pageNo' : page,
        'numOfRows' : 10,
        'type' : 'json',
  }

  response = requests.get(url, params=params)
  try:
    response.raise_for_status()

    items = response.json()['body']['items']
    list = [] # 식품 정보를 관리할 리스트
    labels = {
       '식품명' : 'FOOD_NM_KR',
       '탄수화물(g)' : 'AMT_NUM6',
       '지방(g)' : 'AMT_NUM4',
       '단백질(g)' : 'AMT_NUM3',
       '나트륨(mg)' : 'AMT_NUM13',
       '총칼로리(kal)' : 'AMT_NUM1',
       '기준량' : 'SERVING_SIZE',
    }
    for item in items:
      dic={}
      for column, label in labels.items():
        dic[column] = item[label]
      list.append(dic)
            
    return pd.DataFrame(list)

  except Exception as e:
    print(f'예외 발생 : {e}')
    return pd.DataFrame()
  
print(get_food_nutrient(serviceKey, 2))


           식품명 탄수화물(g)  지방(g) 단백질(g)  나트륨(mg) 총칼로리(kal)   기준량
0        김밥_채소   26.65   3.65   4.60  309.000   158.000  100g
1        김밥_치즈   22.10   7.03   6.24  169.000   177.000  100g
2       김밥_풋고추   27.52   4.41   4.88  327.000   169.000  100g
3        덮밥_낙지   24.12   3.34   5.88  212.000   150.000  100g
4       덮밥_닭고기   14.82   2.18  11.40  153.000   125.000  100g
5  덮밥_돼지고기(제육)   16.86  10.77   9.43  174.000   202.000  100g
6       덮밥_불고기   26.96   5.31   6.60  253.000   182.000  100g
7       덮밥_오징어   21.94   2.01   7.18  162.000   135.000  100g
8          보리밥   36.77   0.24   2.90     4.00   161.000  100g
9          볶음밥   33.97   2.76   5.56  212.000   183.000  100g


In [9]:
def clean_food_nutrient(df : pd.DataFrame)->pd.DataFrame:
  """식품영양소 df에서 결측치 제거, 중복 제거하는 함수"""
  df = df.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거
  # df = df.drop_duplicates(subset=['식품명'], keep='last') # 중복되면 마지막을 살리고 나머지 제거
  # df = df.drop_duplicates(subset=['식품명'], keep=False) # 중복되면 모두 제거

  # 결측치 값 변경
  df.loc[ : ,'탄수화물(g)'] = df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경


  # 결측치 제거
  df = df.dropna() # 행에 결측치가 하나라도 있으면 삭제
  df = df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제

  return df

In [10]:
food_df = get_food_nutrient(serviceKey, 1)
food_df.loc[len(food_df)] = {'식품명' : '국밥_돼지머리'}
food_df = clean_food_nutrient(food_df)
print(food_df)

       식품명 탄수화물(g) 지방(g) 단백질(g)  나트륨(mg) 총칼로리(kal)   기준량
0  국밥_돼지머리   15.94  5.16   6.70  181.000   137.000  100g
1  국밥_순대국밥   10.38  2.28   3.17  126.000     75.00  100g
2   국밥_콩나물   10.93  0.24   1.45  172.000     52.00  100g
3      기장밥   36.77  0.57   3.44     1.00   166.000  100g
4       김밥   19.98  4.55   4.84  307.000   140.000  100g
5    김밥_김치   19.17  4.03   4.30  349.000   130.000  100g
6   김밥_날치알   28.66  4.26   6.10  299.000   177.000  100g
7   김밥_돈가스   31.64  5.81   5.77  241.000   202.000  100g
8   김밥_소고기   25.78  5.56   6.46  267.000   179.000  100g
9    김밥_참치   20.26  7.22   7.00  335.000   174.000  100g


In [8]:
food_df.to_csv(f'food_nutrient.csv', index=False, encoding='utf-8-sig')